# TN4 — Test cuối trên GHIJ, DS-TCN 64 kênh tầm nhìn 121

Train đủ **ABCDEFKL**, chấm **một lần** trên **GHIJ** — 537 buổi ghi của 4 người
chưa từng tham gia bất kỳ bước chọn cấu hình nào. Đây là **số công bố**, theo
`docs/PROTOCOL.md` mục 6.

## Cấu hình

| | |
|---|---|
| model | `ds_tcn --channels 64 --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element` |
| tham số | **38.105** |
| tầm nhìn | **121** — phủ 60% cửa sổ vào, và 121% một pha thở |
| dev, MSE thuần | 0,757855 *(TN2, 4 fold, 1 seed)* |

## Vì sao tầm nhìn 121

TN2 chia bốn mức tầm nhìn thành hai nhóm tách rõ:

| tầm nhìn | c64 | c192 | |
|---:|---:|---:|---|
| 61 | 0,760878 | 0,762714 | nhóm tốt |
| **121** | **0,757855** | **0,764428** | **nhóm tốt** |
| 181 | 0,743657 | 0,732562 | nhóm tệ |
| 241 | 0,736970 | 0,736623 | nhóm tệ |

Hai nhóm cách nhau **0,0142**, lớn hơn dao động giữa các seed (0,0007–0,0108)
nên đọc được. Trong nhóm tốt chỉ chênh **0,0030**, nhỏ hơn dao động seed nên
**không xếp hạng được** — ở c64 thì 61 nhỉnh hơn, ở c192 thì 121 nhỉnh hơn, cả
hai đều dưới nhiễu.

Chọn 121 theo một tiêu chí thiết kế nêu được trước khi nhìn kết quả: nhịp thở
0,25 Hz lấy mẫu 50 Hz nên **một chu kỳ thở đúng 200 mẫu, một pha 100 mẫu**. Tầm
nhìn 61 chỉ phủ 61% một pha, chưa nhìn trọn một lần hít vào; **121 là mức nhỏ
nhất trong các mức đã thử phủ trọn một pha**. Đúng tinh thần Bai et al. mục A.1:
chọn `k` và `d` sao cho tầm nhìn phủ đủ ngữ cảnh mà bài toán cần.

Không viết "tầm nhìn 121 tốt hơn 61" — số liệu ở c64 nói ngược lại.

## Hai nhóm chạy, mỗi nhóm ba seed

**Mục 3 — loss lai alpha 0,0**, tức Pearson thuần. Mức này là **đỉnh TN3 của
chính cấu hình này**: quét mười mức cho `Ours-64/121` thì alpha 0,0 đạt
**0,780306**, cao nhất, hơn MSE thuần 0,0225.

Cùng quy tắc đã dùng cho hai lần chạy TN4 kia — mỗi cấu hình lấy đỉnh TN3 của
chính nó: `Ours-64/61` lấy 0,6 và `Ours-192/121` lấy 0,2.

**Vì sao không lấy 0,6 như `Ours-64/61`.** Ở cấu hình này alpha 0,6 cho
**0,752386 — thấp nhất trong mười mức, và là mức duy nhất trong cả đồ án thua
MSE thuần**. Hai cấu hình chỉ khác nhau tầm nhìn mà đỉnh nhảy từ 0,6 sang 0,0.
Chạy test ở mức đáy sẽ ra một con số thấp một cách vô nghĩa.

Đánh đổi: lần chạy này khác `TN4 Ours-64/61` **hai biến** — tầm nhìn và alpha —
nên không so trực tiếp hai con số GHIJ với nhau được. Phải chọn: hoặc giữ phép
so một biến nhưng chạy ở mức đáy, hoặc dùng đỉnh của chính cấu hình và mất phép
so. Chọn cái sau, vì số công bố quan trọng hơn.

**Mục 4 — MSE thuần.** Cùng kiến trúc, chỉ đổi hàm loss. Đây là thứ đang thiếu ở
mọi cấu hình khác: không có nó thì **không nói được hàm loss lai có giúp trên
tập test hay không**, chỉ nói được là nó giúp trên tập phát triển.

Mỗi lần chạy khoảng **20 phút** *(train 20 epoch ~11,5 phút, chấm 537 buổi ghi
~9 phút)*. Mỗi nhóm ba seed ≈ **1 giờ**, cả hai nhóm ≈ **2 giờ**.

## Một quy tắc phải giữ

`alpha 0,0` đã chốt **trước khi chạy**, lấy từ TN3 trên dev. Nếu về sau
`TN3_HybridLoss_DS_TCN_c64_rf121` chạy xong và cho đỉnh ở mức khác, mà bạn chạy
tiếp mức đó rồi **lấy con số GHIJ cao hơn trong hai cái** — thì GHIJ đã thành
tập chọn cấu hình và toàn bộ giao thức hỏng. Chạy thêm thì phải **báo cáo cả
hai**, kèm lý do vì sao có hai lần chạy.

`run_final_test.py` tự nén và chép sang Drive sau mỗi lần chạy nên không cần ô
lưu riêng. Dừng giữa chừng cũng được: mở lại, chạy ô khôi phục ở mục 1 rồi bấm
tiếp seed còn thiếu.

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn.

In [2]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : b6829a5
commit MobiVital : 4319731 (đã ghim)
GPU              : NVIDIA L4, 23034 MiB


Lấy `by_user/` và `windows/` từ Drive. Test cuối đọc `windows/final_train/`, cắt gộp cả 8 người theo đúng thứ tự MobiVital.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


Khôi phục các seed đã chạy.

**Chạy ô này mỗi khi mở lại notebook.** Mẫu tên tệp bắt riêng `c64_k5_`, không lẫn với tệp của notebook `c64` tầm nhìn 61.

In [4]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn4_*c64_k5_*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

khôi phục 0 dòng vào runs/summary.csv


## 2. Kiểm bản cài đặt

Số tham số phải ra đúng **38.105**.

In [5]:
!python scripts/check_model.py --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   38105

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 5, 4 khối -> tầm nhìn 121, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 121/200 mẫu gần nhất, mất 40% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT 

## 3. Loss lai alpha 0,0 — ba seed

**seed 0**

In [6]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 0

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_pearson_a0_corr0.9_seed0
thiết bị NVIDIA L4

292708 cửa sổ train
38105 tham số

epoch  0  mse 0.41203  pearson 0.5164   0.6 phút
epoch  1  mse 0.38839  pearson 0.5665   1.2 phút
epoch  2  mse 0.37682  pearson 0.5775   1.7 phút
epoch  3  mse 0.37902  pearson 0.5849   2.3 phút
epoch  4  mse 0.38113  pearson 0.5895   2.9 phút
epoch  5  mse 0.38071  pearson 0.5939   3.4 phút
epoch  6  mse 0.38082  pearson 0.5979   4.0 phút
epoch  7  mse 0.38063  pearson 0.6015   4.6 phút
epoch  8  mse 0.37930  pearson 0.6051   5.1 phút
epoch  9  mse 0.38172  pearson 0.6072   5.7 phút
epoch 10  mse 0.38929  pearson 0.6099   6.3 phút
epoch 11  mse 0.39128  pearson 0.6115   6.8 phút
epoch 12  mse 0.40055  pearson 0.6129   7.4 phút
epoch 13  mse 0.41798  pearson 0.6147   8.0 phút
epoch 14  mse 0.42945  pearson 0.6164   8.5 phút
epoch 15  mse 0.43842  pearson 0.6177   9.1 phút
epoch 16  mse 0.45072  pearson 0.6192   9.7 phút
epoch 17  m

**seed 1**

In [7]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 1

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_pearson_a0_corr0.9_seed1
thiết bị NVIDIA L4

292708 cửa sổ train
38105 tham số

epoch  0  mse 0.67948  pearson 0.5163   0.6 phút
epoch  1  mse 0.71874  pearson 0.5622   1.1 phút
epoch  2  mse 0.66624  pearson 0.5733   1.7 phút
epoch  3  mse 0.63723  pearson 0.5805   2.2 phút
epoch  4  mse 0.60184  pearson 0.5861   2.8 phút
epoch  5  mse 0.59361  pearson 0.5903   3.3 phút
epoch  6  mse 0.58383  pearson 0.5937   3.9 phút
epoch  7  mse 0.58873  pearson 0.5975   4.4 phút
epoch  8  mse 0.59402  pearson 0.6009   5.0 phút
epoch  9  mse 0.59354  pearson 0.6038   5.5 phút
epoch 10  mse 0.58594  pearson 0.6066   6.1 phút
epoch 11  mse 0.57805  pearson 0.6096   6.6 phút
epoch 12  mse 0.57643  pearson 0.6121   7.2 phút
epoch 13  mse 0.57257  pearson 0.6145   7.7 phút
epoch 14  mse 0.57640  pearson 0.6163   8.3 phút
epoch 15  mse 0.58482  pearson 0.6186   8.8 phút
epoch 16  mse 0.58948  pearson 0.6199   9.4 phút
epoch 17  m

**seed 2**

In [8]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse_pearson --alpha 0.0 --seed 2

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_pearson_a0_corr0.9_seed2
thiết bị NVIDIA L4

292708 cửa sổ train
38105 tham số

epoch  0  mse 0.54338  pearson 0.5177   0.6 phút
epoch  1  mse 0.56919  pearson 0.5618   1.1 phút
epoch  2  mse 0.54964  pearson 0.5746   1.7 phút
epoch  3  mse 0.53559  pearson 0.5825   2.2 phút
epoch  4  mse 0.54043  pearson 0.5873   2.8 phút
epoch  5  mse 0.54439  pearson 0.5915   3.3 phút
epoch  6  mse 0.54813  pearson 0.5953   3.9 phút
epoch  7  mse 0.55001  pearson 0.5985   4.4 phút
epoch  8  mse 0.55187  pearson 0.6017   5.0 phút
epoch  9  mse 0.55473  pearson 0.6044   5.6 phút
epoch 10  mse 0.55602  pearson 0.6066   6.1 phút
epoch 11  mse 0.55732  pearson 0.6085   6.7 phút
epoch 12  mse 0.55431  pearson 0.6115   7.2 phút
epoch 13  mse 0.54523  pearson 0.6140   7.8 phút
epoch 14  mse 0.55277  pearson 0.6152   8.3 phút
epoch 15  mse 0.55188  pearson 0.6174   8.9 phút
epoch 16  mse 0.56053  pearson 0.6188   9.4 phút
epoch 17  m

## 4. MSE thuần — ba seed

Cùng kiến trúc, chỉ đổi hàm loss. Hiệu giữa mục 3 và mục 4 là **đóng góp của hàm loss lai đo trên tập test**.

**seed 0**

In [9]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse --seed 0

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị NVIDIA L4

292708 cửa sổ train
38105 tham số

epoch  0  mse 0.05017  pearson 0.4067   0.5 phút
epoch  1  mse 0.02822  pearson 0.5334   1.0 phút
epoch  2  mse 0.02579  pearson 0.5592   1.5 phút
epoch  3  mse 0.02478  pearson 0.5685   2.0 phút
epoch  4  mse 0.02431  pearson 0.5727   2.5 phút
epoch  5  mse 0.02391  pearson 0.5766   3.0 phút
epoch  6  mse 0.02371  pearson 0.5798   3.5 phút
epoch  7  mse 0.02346  pearson 0.5822   4.0 phút
epoch  8  mse 0.02325  pearson 0.5845   4.5 phút
epoch  9  mse 0.02307  pearson 0.5870   5.0 phút
epoch 10  mse 0.02294  pearson 0.5887   5.5 phút
epoch 11  mse 0.02274  pearson 0.5900   6.0 phút
epoch 12  mse 0.02265  pearson 0.5908   6.5 phút
epoch 13  mse 0.02254  pearson 0.5919   7.0 phút
epoch 14  mse 0.02240  pearson 0.5941   7.6 phút
epoch 15  mse 0.02229  pearson 0.5945   8.1 phút
epoch 16  mse 0.02220  pearson 0.5955   8.6 phút
epoch 17  mse 0.02210 

**seed 1**

In [10]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse --seed 1

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed1
thiết bị NVIDIA L4

292708 cửa sổ train
38105 tham số

epoch  0  mse 0.05576  pearson 0.3878   0.5 phút
epoch  1  mse 0.02755  pearson 0.5266   1.0 phút
epoch  2  mse 0.02534  pearson 0.5554   1.5 phút
epoch  3  mse 0.02452  pearson 0.5663   2.0 phút
epoch  4  mse 0.02400  pearson 0.5722   2.6 phút
epoch  5  mse 0.02366  pearson 0.5773   3.1 phút
epoch  6  mse 0.02347  pearson 0.5808   3.6 phút
epoch  7  mse 0.02322  pearson 0.5839   4.1 phút
epoch  8  mse 0.02302  pearson 0.5865   4.6 phút
epoch  9  mse 0.02293  pearson 0.5878   5.1 phút
epoch 10  mse 0.02271  pearson 0.5908   5.6 phút
epoch 11  mse 0.02259  pearson 0.5921   6.1 phút
epoch 12  mse 0.02245  pearson 0.5944   6.6 phút
epoch 13  mse 0.02234  pearson 0.5948   7.1 phút
epoch 14  mse 0.02224  pearson 0.5967   7.6 phút
epoch 15  mse 0.02213  pearson 0.5976   8.1 phút
epoch 16  mse 0.02203  pearson 0.5980   8.6 phút
epoch 17  mse 0.02196 

**seed 2**

In [11]:
!python scripts/run_final_test.py --experiment tn4 --model ds_tcn --channels 64 \
    --kernel_size 5 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element \
    --loss mse --seed 2

thực nghiệm tn4  -> runs/tn4/
run_id   ds_tcn_c64_k5_n4_none_do0.2_dpel_mse_corr0.9_seed2
thiết bị NVIDIA L4

292708 cửa sổ train
38105 tham số

epoch  0  mse 0.05406  pearson 0.4110   0.5 phút
epoch  1  mse 0.02767  pearson 0.5362   1.0 phút
epoch  2  mse 0.02573  pearson 0.5581   1.5 phút
epoch  3  mse 0.02480  pearson 0.5673   2.0 phút
epoch  4  mse 0.02427  pearson 0.5721   2.5 phút
epoch  5  mse 0.02391  pearson 0.5774   3.0 phút
epoch  6  mse 0.02363  pearson 0.5810   3.5 phút
epoch  7  mse 0.02345  pearson 0.5838   4.0 phút
epoch  8  mse 0.02328  pearson 0.5859   4.5 phút
epoch  9  mse 0.02309  pearson 0.5884   5.0 phút
epoch 10  mse 0.02291  pearson 0.5901   5.5 phút
epoch 11  mse 0.02281  pearson 0.5919   6.0 phút
epoch 12  mse 0.02263  pearson 0.5928   6.5 phút
epoch 13  mse 0.02257  pearson 0.5943   7.0 phút
epoch 14  mse 0.02246  pearson 0.5949   7.5 phút
epoch 15  mse 0.02236  pearson 0.5961   8.0 phút
epoch 16  mse 0.02225  pearson 0.5971   8.5 phút
epoch 17  mse 0.02219 

## 5. Kết quả

`compare_cv.py --final` gộp các seed của cùng một cấu hình thành `mean ± std`.
Nó nhận ra các lần chạy cùng cấu hình bằng cách bỏ hậu tố `_seed<N>` khỏi
`run_id`, nên hai nhóm ở mục 3 và mục 4 tự tách thành hai dòng.

Cột điểm là **macro** — trung bình theo người. Muốn xem cả **micro** thì mở
`notebooks/BANG_DIEM_MICRO_MACRO.ipynb`, nó đọc thẳng `score_micro` từ Drive.

**Mốc đối chiếu trên GHIJ**, cùng pipeline và cùng ba seed:

| | tham số | macro |
|---|---:|---:|
| LSTM-352 *(kiến trúc MobiVital)* | 1.502.713 | 0,810302 ± 0,015403 |
| Ours-64/61 alpha 0,6 | 37.081 | 0,803590 ± 0,015350 |
| Ours-192/121 alpha 0,2 | 310.873 | 0,800721 ± 0,010705 |
| DS-TCN-nền, MSE | 56.281 | 0,795782 ± 0,015413 |

In [ ]:
!python scripts/compare_cv.py --experiment tn4 --final

## 6. Ngắt phiên

In [ ]:
from google.colab import runtime
runtime.unassign()